# GeoGuard — PoC Notebook
Team: GeoGuard · Country: Yemen · Pilot: UAE

This submission runs from bundled inputs to a website-compatible result without credentials,
network queries, model downloads or manual widget clicks. Run every cell in order.

It recomputes NO₂, CO, SO₂ and regional CH₄ daily/monthly/period summaries from actual Earth Engine exports
and packages a previously executed, dated methane result. It does not rerun methane
inference. Use `Geoguard.ipynb` for new imagery and interactive model checks.

Atmospheric columns, regional methane and detailed methane candidates remain separate. Neither proves overall air safety.

In [1]:
from pathlib import Path
import json, sys, math, csv, base64
root = Path.cwd().resolve()
if not (root / 'geoguard').is_dir() and (root.parent / 'geoguard').is_dir():
    root = root.parent
assert (root / 'geoguard').is_dir(), 'Run from the cloned Geoguard repository.'
sys.path.insert(0, str(root))
from geoguard.no2 import unpack_stacked, summarize
from geoguard.measurements import summarize_export, METRICS
from geoguard.core import SITES, public_methane, export_bundle
config = json.loads((root / 'examples/input.json').read_text(encoding='utf-8'))
print(json.dumps(config, indent=2))

{
  "start": "2026-01-01",
  "end_exclusive": "2026-09-01",
  "radius_m": 5000,
  "collection": "COPERNICUS/S5P/NRTI/L3_NO2",
  "no2_source": "sources/no2_earthengine_export.geojson",
  "methane_source": "docs/data/demo.json",
  "mode": "recompute_saved_satellite_measurements_and_replay_dated_methane",
  "measurements_source": "sources/measurements_earthengine_export.geojson",
  "measurements_start": "2026-08-01",
  "measurements_end_exclusive": "2026-10-01"
}


## 1. Read the example input and calculate NO₂ history
The source contains daily spatial reductions for two 5 km circles, January–August 2026.
Missing days stay unknown. Negative valid retrievals are retained. Each usable day
has equal weight in the period mean. Units are mol/m² through the atmospheric column,
not ground-level concentration or AQI.

In [2]:
raw = json.loads((root / config['no2_source']).read_text(encoding='utf-8'))
rows = unpack_stacked(raw['features'], config['start'], config['end_exclusive'])
no2 = summarize(rows, SITES, config['start'], config['end_exclusive'],
                config['radius_m'], config['collection'])
for site in no2['sites']:
    print(f"{site['name']}: {site['mean_mol_m2']:.12g} mol/m²; "
          f"{site['valid_days']}/{site['total_days']} usable days")

Dubai comparison area: 0.000207851401957 mol/m²; 226/243 usable days
Jebel Ali comparison area: 0.000206665923533 mol/m²; 224/243 usable days


## 1A. Calculate the additional feasible satellite measurements
CO and SO₂ are atmospheric columns; regional CH₄ is a column-averaged dry-air
mixing ratio in ppb. These measured quantities use August–September 2026,
not the January–August NO₂ window. The raw source includes their identities,
coordinates, radius, dates and valid-pixel counts. CH₄ uses OFFL only.
PM, VOC and H₂S measurements require inputs or sensors that have not been connected.
Satellite aerosol index is an indirect aerosol indicator, not measured PM mass.

In [3]:
raw_extra = json.loads((root / config['measurements_source']).read_text(encoding='utf-8'))
measurements = {}
for metric in ['CO', 'SO2', 'CH4']:
    features = [f for f in raw_extra['features'] if f['properties']['metric'] == metric]
    value = summarize_export(features, metric, SITES, config['measurements_start'],
        config['measurements_end_exclusive'], config['radius_m'], 'OFFL' if metric == 'CH4' else 'NRTI')
    measurements[metric] = value
    spec = METRICS[metric]
    for site in value['sites']:
        mean = site['mean_value']
        print(metric, site['name'], None if mean is None else mean * spec['display_factor'],
            spec['display_units'], f"{site['valid_days']}/{site['total_days']} usable days")

CO Dubai comparison area 36.33414750830455 mmol/m² 52/61 usable days
CO Jebel Ali comparison area 37.6982856798005 mmol/m² 60/61 usable days
SO2 Dubai comparison area 145.82788242312893 µmol/m² 60/61 usable days
SO2 Jebel Ali comparison area 143.9617467042297 µmol/m² 60/61 usable days
CH4 Dubai comparison area 1966.9282903751287 ppb 33/61 usable days
CH4 Jebel Ali comparison area 1969.5609751337006 ppb 37/61 usable days


## 2. Attach the existing methane evidence
This is a replay of the UAE check observed on 22 September 2026 against 9 September,
at 23.86479° N, 53.61893° E. It is a different location and date from the NO₂ study.
Zero candidate pixels does not mean zero methane. 96.04% is usable coverage, not accuracy.
No new model inference or forecast is claimed.

In [4]:
saved = json.loads((root / config['methane_source']).read_text(encoding='utf-8'))
figure = 'data:image/png;base64,' + base64.b64encode((root / 'docs/assets/uae-example.png').read_bytes()).decode()
methane = public_methane(saved['methane'], figure)
print(json.dumps({k:v for k,v in methane.items() if k != 'figure_data'}, indent=2, ensure_ascii=False))

{
  "status": "no_candidate",
  "message": "No methane-like candidate passed the rule in the saved 22 September 2026 image. This is not a clean-air conclusion.",
  "checked_at": "2026-09-22T22:47:59.698378+00:00",
  "current_acquired_at": "2026-09-22T07:12:57.703362Z",
  "background_acquired_at": "2026-09-09T07:13:12.928948Z",
  "current_product_id": "S2B_MSIL1C_20260922T065619_N0513_R063_T39QYG_20260922T104758",
  "background_product_id": "S2A_MSIL1C_20260909T070321_N0512_R063_T39QYG_20260909T104214",
  "provider_available_at": "2026-09-22T11:37:44.508375Z",
  "latitude": 23.86479,
  "longitude": 53.61893,
  "requested_bounds": [
    53.609112391785885,
    23.85576116054776,
    53.62874760821411,
    23.87381882886855
  ],
  "pipeline_version": "asap-s2-v2",
  "model_id": "MARS-S2L/MARSS2L_20250326",
  "weights_sha256": "be634fb9e24dc4877f44c1ff9f69972e6f0453e30d70c0dc03677876340ef246",
  "candidate_pixels": 0,
  "common_valid_fraction": 0.9604,
  "history_count": 1,
  "forecast": {

## 3. Verify the numerical result against the committed example
The expected file is supplied for comparison. The calculations above use the raw
GeoJSON, not the expected means. Checks cover daily and monthly parity, missing-day
counts, and the location/date of the methane record.

In [5]:
expected = json.loads((root / 'examples/expected_summary.json').read_text(encoding='utf-8'))
assert no2['sites'] == saved['no2']['sites'], 'Daily/monthly values differ from archived evidence'
for metric, value in measurements.items():
    assert value['sites'] == saved['measurements'][metric]['sites']
    assert value['units'] == METRICS[metric]['units']
    for site, reference in zip(value['sites'], expected['measurements'][metric]):
        assert site['id'] == reference['id']
        assert math.isclose(site['mean_value'], reference['mean_value'], rel_tol=1e-12)
        assert site['valid_days'] == reference['valid_days']
        assert site['total_days'] == reference['total_days']
    # Maps are archived real Earth Engine outputs; recalculation above is numerical.
    value['map'] = saved['measurements'][metric]['map']
    value['source_export'] = config['measurements_source']
    value['retrieved_at'] = saved['measurements'][metric]['retrieved_at']
for site, reference in zip(no2['sites'], expected['no2']):
    assert site['id'] == reference['id']
    assert math.isclose(site['mean_mol_m2'], reference['mean_mol_m2'], rel_tol=1e-12)
    assert site['valid_days'] == reference['valid_days']
    assert site['total_days'] == reference['total_days']
for key, value in expected['methane'].items():
    assert methane[key] == value, key
print('Example checks passed.')

Example checks passed.


## 4. Export the output
`outputs/submission/result.json` can be imported into the demo website.
The CSV provides a compact table. Generation timestamps change each run;
the source observation dates and numerical results do not.

In [6]:
output = root / 'outputs/submission'
output.mkdir(parents=True, exist_ok=True)
result = export_bundle(output / 'result.json', no2, methane, measurements)
with (output / 'no2_summary.csv').open('w', newline='', encoding='utf-8') as handle:
    writer = csv.DictWriter(handle, fieldnames=['id', 'mean_mol_m2', 'valid_days', 'total_days'])
    writer.writeheader()
    writer.writerows({key: site[key] for key in writer.fieldnames} for site in no2['sites'])
with (output / 'measurements_summary.csv').open('w', newline='', encoding='utf-8') as handle:
    fields = ['metric', 'id', 'mean_value', 'units', 'start', 'end_exclusive', 'valid_days', 'total_days']
    writer = csv.DictWriter(handle, fieldnames=fields)
    writer.writeheader()
    for metric, value in measurements.items():
        for site in value['sites']:
            writer.writerow({**{k: site[k] for k in ['id', 'mean_value', 'valid_days', 'total_days']},
                **{k: value[k] for k in ['units', 'start', 'end_exclusive']}, 'metric': metric})
assert json.loads(result.read_text(encoding='utf-8'))['schema'] == 'geoguard-demo-v1'
print('Complete: result.json, no2_summary.csv and measurements_summary.csv')

Complete: result.json, no2_summary.csv and measurements_summary.csv


## Interpretation and next steps
The two close NO₂ averages alone do not establish a significant difference or a source.
Clouds, observation dates and retrieval uncertainty affect comparisons. The saved methane
record is evidence about one screened observation, not an air-quality verdict.

For fresh queries, open `Geoguard.ipynb`: Earth Engine sign-in is required for fresh satellite
queries; the experimental methane workflow downloads public imagery and pretrained weights.
See `README.md`, `VALIDATION.md` and `THIRD_PARTY_NOTICES.md` for scope and provenance.